# Zajęcia 3 — dane: pozyskanie, wersjonowanie, kontrakt i governance

Model jest funkcją danych. Dopóki nie wiemy dokładnie, **których** danych, nie da się powiedzieć, co właściwie zbudowaliśmy.

W tym notebooku:

1. pobieramy zbiór **ze zdalnego źródła** i zapisujemy jego tożsamość,
2. opisujemy go manifestem z sumą kontrolną,
3. wersjonujemy plik przez **DVC (Data Version Control, kontrola wersji danych)**, porównujemy trening na dwóch wersjach danych i odtwarzamy pierwszy wynik w świeżym klonie,
4. definiujemy **kontrakt danych** i sprawdzamy, co się dzieje, gdy zostanie złamany,
5. szukamy wycieku etykiety,
6. spisujemy metryczkę zbioru.

Komórki kodu opisane jako **zadanie** wypełniasz samodzielnie; komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`. Polecenia DVC wykonujesz samodzielnie w terminalu, zgodnie z instrukcjami w komórkach Markdown.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import hashlib
import json
import sys
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
import pandas as pd
import pandera.pandas as pa
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE = REPO_ROOT / "artifacts" / "zajecia_03"
PROJECT_DIR = BASE / "projekt"
STORAGE_DIR = BASE / "magazyn"
REPORTS_DIR = BASE / "reports"
for folder in (PROJECT_DIR / "data", STORAGE_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
DATA_PATH = PROJECT_DIR / "data" / "iris.csv"
MANIFEST_PATH = PROJECT_DIR / "data" / "iris.manifest.json"


def write_file(path: Path, content: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Zapisano: {path.name}")
    return path


print(f"Katalog projektu: {PROJECT_DIR.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Pozyskanie zbioru ze zdalnego źródła

Dane pobieramy z konkretnego adresu, a nie z funkcji bibliotecznej. To jest sytuacja, którą spotkasz w projekcie: plik leży gdzieś na zewnątrz, ktoś inny nim zarządza i może go zmienić bez uprzedzenia.

**Zadanie:** napisz `fetch_dataset(url, destination)`, która pobiera plik, ponawia próbę przy błędzie sieci i **przy trwałym niepowodzeniu przechodzi na źródło zapasowe** zamiast przerywać pracę. Funkcja ma zwrócić informację, skąd faktycznie pochodzą dane.

In [ ]:
# Zadanie: napisz pobieranie danych ze zdalnego źródła.


In [ ]:
%%skip True
COLUMNS = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]


def _fallback_frame() -> pd.DataFrame:
    """Zrodlo zapasowe: ten sam zbior z biblioteki, sprowadzony do tych samych kolumn."""
    from sklearn.datasets import load_iris

    bundle = load_iris(as_frame=True)
    frame = bundle.frame.copy()
    frame.columns = COLUMNS[:4] + ["target"]
    frame["species"] = [bundle.target_names[value] for value in frame["target"]]
    return frame[COLUMNS]


def fetch_dataset(url: str, destination: Path, attempts: int = 3) -> dict:
    destination.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, attempts + 1):
        try:
            response = httpx.get(url, timeout=20.0, follow_redirects=True)
            response.raise_for_status()
            destination.write_bytes(response.content)
            return {"source": url, "origin": "zdalne", "attempts": attempt}
        except Exception as exc:
            print(f"  próba {attempt}/{attempts} nieudana: {type(exc).__name__}")
            if attempt < attempts:
                time.sleep(2**attempt)

    _fallback_frame().to_csv(destination, index=False)
    return {"source": "sklearn.datasets.load_iris", "origin": "zapasowe", "attempts": attempts}


provenance = fetch_dataset(DATA_URL, DATA_PATH)
raw = pd.read_csv(DATA_PATH)

print(f"\nŹródło: {provenance['origin']} ({provenance['source']})")
print(f"Wierszy: {len(raw)}, kolumn: {len(raw.columns)}")
display(raw.head(3))

## 3. Manifest zbioru

Plik danych bez opisu jest anonimowy. Manifest odpowiada na cztery pytania: **skąd**, **kiedy**, **co dokładnie** i **jak sprawdzić, że to wciąż to samo**.

Ostatnie pytanie rozstrzyga suma kontrolna. Nie liczymy jej „na wszelki wypadek" — liczymy ją po to, żeby przy każdym kolejnym uruchomieniu móc **odmówić pracy** na innych danych.

### Czym jest SHA i dlaczego zapisujemy je w manifeście?

**SHA** (*Secure Hash Algorithm*) to rodzina kryptograficznych funkcji skrótu. Tutaj używamy **SHA-256**: algorytm czyta bajty pliku i wylicza z nich 256-bitowy skrót, zapisany w polu `sha256` jako **64 znaki szesnastkowe**. To „odcisk palca” zawartości pliku, a nie szyfrowanie — skrót nie służy do odtworzenia danych.

Ten sam plik zawsze daje ten sam skrót. Zmiana choćby jednej wartości, kolejności wierszy lub znaku końca linii z praktyczną pewnością zmieni wynik. Zmiana samej nazwy lub lokalizacji pliku nie zmienia skrótu. Porównujemy więc **dokładną zawartość pliku**, a nie tylko nazwę, liczbę wierszy czy typy kolumn. Dwa różne pliki teoretycznie mogą mieć ten sam skrót (kolizja), ale dla SHA-256 ryzyko przypadkowej kolizji jest pomijalne.

**Po co dodajemy `sha256` do manifestu?** Adres źródła mówi, skąd pobrano dane, lecz zawartość pod tym adresem może się zmienić. Skrót zapisuje punkt odniesienia dla konkretnej pobranej wersji. Dzięki niemu możemy:

- **Sprawdzić integralność** — wykryć uszkodzenie pobranego pliku albo jego późniejszą edycję, nawet gdy rozmiar i schemat pozostały takie same.
- **Odtworzyć eksperyment** — potwierdzić, że trening lub ewaluacja korzystają z identycznego pliku jak poprzednio, i powiązać wyniki z wersją danych.
- **Zatrzymać pipeline przy niezgodności** — przed użyciem danych ponownie obliczyć skrót i porównać go z wartością w zachowanym manifeście. W następnej sekcji robi to `verify_dataset`.
- **Rozpoznawać identyczne pliki** — np. wykrywać duplikaty lub identyfikować dane w pamięci podręcznej niezależnie od nazwy pliku.

**Ważne:** manifest referencyjny trzeba zachować, np. wersjonować razem z eksperymentem. Wygenerowanie go od nowa z podmienionych danych przed weryfikacją zatwierdzi nową zawartość zamiast wykryć zmianę. Sam SHA-256 nie dowodzi też, że dane są poprawne, legalnie pozyskane lub pochodzą od zaufanego autora. Jeśli ktoś może podmienić zarówno plik, jak i manifest, samo porównanie skrótów nie zapewni autentyczności.

**Zadanie:** napisz `build_manifest(path, provenance)` tworzącą opis zbioru i zapisz go obok pliku danych.

In [ ]:
# Zadanie: zbuduj manifest zbioru.


In [ ]:
%%skip True
def sha256_of(path: Path) -> str:
    # SHA-256 wylicza jeden skrót z całej zawartości pliku, a nie z jego nazwy.
    # Czytamy surowe bajty (rb) porcjami po 1 MiB (1 << 20), aż do b"" (końca pliku),
    # dzięki czemu nie trzeba wczytywać całego pliku do pamięci.
    # update dołącza kolejne porcje do obliczeń tego samego skrótu;
    # podział na porcje nie zmienia wyniku. hexdigest zapisuje go jako 64 znaki szesnastkowe.
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_manifest(path: Path, provenance: dict) -> dict:
    frame = pd.read_csv(path)
    manifest = {
        "name": path.stem,
        "source": provenance["source"],
        "origin": provenance["origin"],
        "retrieved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "sha256": sha256_of(path),
        "size_bytes": path.stat().st_size,
        "rows": int(len(frame)),
        "columns": {name: str(dtype) for name, dtype in frame.dtypes.items()},
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


manifest = build_manifest(DATA_PATH, provenance)
print(json.dumps(manifest, indent=2))

## 4. Weryfikacja pliku względem manifestu

Manifest, którego nikt nie sprawdza, jest komentarzem. Weryfikacja ma być **pierwszym krokiem potoku** i ma go zatrzymywać.

**Zadanie:** napisz `verify_dataset(path, manifest_path)` zgłaszającą wyjątek, gdy plik nie odpowiada manifestowi. Sprawdź rozmiar, sumę kontrolną i zestaw kolumn.

In [ ]:
# Zadanie: napisz weryfikację pliku względem manifestu.


In [ ]:
%%skip True
class DatasetMismatch(RuntimeError):
    """Plik danych nie odpowiada temu, co opisuje manifest."""


def verify_dataset(path: Path, manifest_path: Path) -> dict:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    problems = []

    if not path.exists():
        raise DatasetMismatch(f"Brak pliku danych: {path.name}")

    actual_size = path.stat().st_size
    if actual_size != manifest["size_bytes"]:
        problems.append(f"rozmiar {actual_size} zamiast {manifest['size_bytes']}")

    actual_hash = sha256_of(path)
    if actual_hash != manifest["sha256"]:
        problems.append(f"suma kontrolna {actual_hash[:12]} zamiast {manifest['sha256'][:12]}")

    actual_columns = list(pd.read_csv(path, nrows=0).columns)
    if actual_columns != list(manifest["columns"]):
        problems.append(f"kolumny {actual_columns}")

    if problems:
        raise DatasetMismatch("Dane nie odpowiadają manifestowi: " + "; ".join(problems))

    print(f"Weryfikacja OK — {manifest['rows']} wierszy, suma {manifest['sha256'][:12]}...")
    return manifest


verify_dataset(DATA_PATH, MANIFEST_PATH)

## 5. Punkt kontrolny 1 — podmieniony plik danych

Zmieniamy jedną wartość w pliku. Nie zmienia się liczba wierszy, liczba kolumn ani nazwa pliku — zmienia się jedna liczba.

**Do opisania:** czy bez sumy kontrolnej ktokolwiek zauważyłby tę zmianę? Po czym?

In [ ]:
original_bytes = DATA_PATH.read_bytes()

tampered = pd.read_csv(DATA_PATH)
print(f"Wartość przed zmianą: {tampered.loc[0, 'sepal_length']}")
tampered.loc[0, "sepal_length"] = 5.2
tampered.to_csv(DATA_PATH, index=False)

try:
    verify_dataset(DATA_PATH, MANIFEST_PATH)
except DatasetMismatch as error:
    print(f"\nZATRZYMANO: {error}")
finally:
    DATA_PATH.write_bytes(original_bytes)

print("\nPo przywróceniu pliku:")
verify_dataset(DATA_PATH, MANIFEST_PATH)

## 6. Wersjonowanie danych — DVC

Suma kontrolna mówi, czy dane się zmieniły. Nie mówi, **jak wrócić** do poprzedniej wersji.

DVC (Data Version Control) to narzędzie do wersjonowania danych i artefaktów, np. wytrenowanych modeli, oraz opisywania potoków obliczeń. Jest dostępne jako pakiet Pythona, ale na co dzień korzystasz z niego przede wszystkim przez polecenie `dvc` w terminalu. **Nie zastępuje Git** — uzupełnia go o obsługę zawartości, której nie chcemy przechowywać w historii Git.

`dvc add` zapisuje plik w lokalnej pamięci podręcznej (cache) i tworzy mały plik `.dvc` ze skrótem oraz metadanymi. Dopiero `dvc push` wysyła zawartość do skonfigurowanego magazynu. Git wersjonuje **wskaźnik**, DVC przechowuje **zawartość**.

```text
repozytorium Git          magazyn DVC
  data/iris.csv.dvc  ───▶  obiekt identyfikowany skrótem
  (kilkaset bajtów)        (właściwy plik)
```

Repozytorium Git to katalog projektu z historią zarządzaną przez Git; w jego katalogu głównym zwykle znajduje się ukryty katalog `.git`. Powstaje on po `git init` albo podczas `git clone`. W terminalu przechodzisz przez `cd` do **katalogu głównego projektu**, nie do wnętrza `.git`, i tam uruchamiasz `dvc init`.

Przykładowy układ po inicjalizacji DVC i dodaniu danych:

```text
moj-projekt/
  .git/                  historia kodu i wskaźników
  .dvc/
    config               konfiguracja DVC, m.in. adres magazynu
    cache/               lokalne obiekty danych; ignorowane przez Git
  .dvcignore             reguły pomijania plików przez DVC
  train.py               kod treningu śledzony przez Git
  data/
    .gitignore           reguła wykluczająca iris.csv z Git
    iris.csv             dane robocze; nie trafiają do Git
    iris.csv.dvc         mały wskaźnik śledzony przez Git
```

**`dvc init` nie wykonuje `git init` i nie tworzy drugiego repozytorium Git.** Dodaje konfigurację DVC do istniejącego projektu. W tym notebooku używamy `dvc init --no-scm`, aby ćwiczyć w odizolowanym katalogu `artifacts/zajecia_03/projekt` bez integracji z historią Git repozytorium zajęć. Ten wariant nie pokazuje jeszcze wspólnego wersjonowania kodu i wskaźników; w zwykłym projekcie stosujesz `dvc init` bez tej opcji.


Plik `iris.csv.dvc` jest zwykłym plikiem tekstowym. Dodajesz go przez `git add` i zapisujesz przez `git commit`, podobnie jak `train.py`. **DVC nie robi tych commitów automatycznie.** Jeden commit może więc wskazywać zarówno konkretny kod treningu, jak i konkretną wersję danych:

| Commit Git | Kod zapisany w commicie | Wskaźnik zapisany w commicie | Zawartość poza Git |
|---|---|---|---|
| Wersja 1 | `train.py` z pierwszego eksperymentu | `iris.csv.dvc` ze skrótem A | Obiekt danych A w magazynie DVC |
| Wersja 2 | `train.py` po zmianach | `iris.csv.dvc` ze skrótem B | Obiekt danych B w magazynie DVC |

Git przechowuje obie wersje **wskaźnika**, nie obie kopie CSV. Po wybraniu starego commita DVC odczytuje stary wskaźnik i potrafi przywrócić odpowiadające mu dane. Warunek: wskazany obiekt nadal musi istnieć w cache lub magazynie. Sam commit Git nie jest kopią zapasową danych.

Nie utożsamiaj skrótu z pliku `.dvc` z naszym `sha256` w manifeście. DVC dobiera własny mechanizm identyfikacji obiektów (dla lokalnych plików zwykle MD5); manifest SHA-256 jest osobnym sprawdzeniem tożsamości danych w tym ćwiczeniu.

### Jak wygląda codzienna praca?

**Poniższe przykłady objaśniają pracę we własnym projekcie — nie wykonuj ich w katalogu ćwiczeniowym.** Zakładają istniejące repozytorium Git z plikami `train.py` i `data/iris.csv`, aktywne środowisko z DVC oraz terminal w katalogu głównym tego projektu. Pełny scenariusz wdrożenia do własnego projektu znajdziesz także w sekcji 13.

**1. Jednorazowo: inicjalizacja i pierwsza wersja danych.**

```bash
dvc init
dvc add data/iris.csv
dvc remote add -d magazyn ../moj-projekt-dane
dvc push
git add .dvc .dvcignore data/.gitignore data/iris.csv.dvc train.py
git status
git commit -m "Pierwsza wersja kodu i danych"
git push
```

- `dvc init` tworzy konfigurację DVC. `dvc add` zapisuje zawartość w cache, tworzy wskaźnik i w trybie integracji z Git dopisuje CSV do `.gitignore`.
- `dvc remote add -d magazyn ../moj-projekt-dane` nadaje magazynowi nazwę `magazyn` i ustawia go jako domyślny (`-d`). Tutaj to katalog obok projektu; w zespole może to być np. S3, Azure Blob albo serwer SSH. Samo ustawienie adresu nie przesyła danych.
- `dvc push` wysyła obiekty danych do magazynu. `git add` przygotowuje kod, konfigurację i wskaźnik do commita. DVC tworzy reguły ignorowania swojego cache, więc zwykłe `git add .dvc` nie dodaje jego zawartości. Nie wymuszaj dodania ignorowanych plików przez `git add -f`.
- `git status` pozwala sprawdzić przygotowane zmiany przed `git commit`. `git commit` zapisuje lokalną wersję, a `git push` wysyła commity do wcześniej skonfigurowanego zdalnego repozytorium Git. **`git push` nie wysyła danych DVC, a `dvc push` nie wysyła commitów.**

**2. Po otrzymaniu nowego eksportu danych.** Zapisujesz nowy CSV pod tą samą ścieżką, w razie potrzeby aktualizujesz kod i wykonujesz:

```bash
dvc add data/iris.csv
dvc push
git add data/iris.csv.dvc train.py
git status
git commit -m "Nowy eksport danych i dostosowanie treningu"
git push
```

`dvc add` aktualizuje wskaźnik i zapisuje nowy obiekt w cache. Commit wiąże ten wskaźnik z kodem. Starsze obiekty mogą pozostać w magazynie, dzięki czemu starsze commity nadal da się odtworzyć. Manifest i inne zmienione metadane również aktualizujesz oraz zapisujesz w tym commicie, jeśli projekt ich używa.

**3. Inna osoba pobiera projekt.**

```bash
git clone ADRES_REPOZYTORIUM moj-projekt
cd moj-projekt
dvc pull
```

`ADRES_REPOZYTORIUM` zastępujesz rzeczywistym adresem. `git clone` pobiera kod, konfigurację i wskaźniki; `dvc pull` pobiera brakujące dane do cache i odtwarza pliki robocze. Przed `dvc pull` potrzebujesz środowiska z DVC oraz dostępu do magazynu. Katalog `../moj-projekt-dane` z pierwszego przykładu nie pojawi się automatycznie na innym komputerze — zespół potrzebuje dostępnego dla wszystkich magazynu i odpowiednich uprawnień. Przy aktualizacji istniejącej kopii zwykle wykonujesz `git pull`, a potem `dvc pull`, po zabezpieczeniu własnych zmian.

**4. Powrót do wersji historycznej.** Najpierw zabezpiecz własne zmiany kodu i danych: niezapisany CSV może zostać nadpisany przy odtwarzaniu. Nowe dane zachowaj przez `dvc add`, `dvc push` i commit wskaźnika — sam `git stash` nie zabezpieczy CSV ignorowanego przez Git. Następnie:

```bash
git switch --detach IDENTYFIKATOR_COMMITA
dvc pull
```

`IDENTYFIKATOR_COMMITA` zastępujesz wybranym hashem z historii Git. `git switch --detach` wybiera ten commit bez przesuwania gałęzi — odtwarza kod i wskaźniki, ale sam nie przełącza ignorowanego CSV. `dvc pull` dobiera dane do wybranego wskaźnika. Jeśli odpowiedni obiekt jest już w lokalnym cache, wystarczy `dvc checkout`: odtwarza pliki z cache, bez pobierania ich z magazynu. Powrót do bieżącej gałęzi również wymaga ponownego dopasowania danych przez `dvc pull` lub `dvc checkout`.

### Co jeszcze warto wiedzieć o DVC?

- **Cache a magazyn:** cache jest lokalną pamięcią roboczą DVC, a magazyn jest osobnym miejscem przechowywania i współdzielenia obiektów. Na zajęciach oba są na jednym dysku — to demonstracja, nie zabezpieczenie przed awarią tego dysku.
- **Diagnostyka:** `dvc status` pokazuje lokalne zmiany danych lub etapów, `dvc status -c` porównuje stan cache i magazynu dla bieżącego projektu, `dvc remote list` pokazuje adresy magazynów, a `dvc diff` pozwala porównać metadane wersji danych (np. bieżący stan z ostatnim commitem). DVC nie wyjaśni za Ciebie znaczenia zmienionych rekordów i nie zastępuje walidacji danych.
- **Potoki:** w `dvc.yaml` można zdefiniować etapy, np. przygotowanie danych, trening i ewaluację, wraz z ich zależnościami i wynikami. `dvc repro` uruchamia etapy wymagające przeliczenia; `dvc.lock` utrwala ich parametry i skróty zależności oraz wyników. Oba pliki wersjonuje się w Git. To kolejna funkcja narzędzia — samo `dvc add` nie tworzy jeszcze potoku.
- **Odtwarzalność ma więcej warunków:** zgodność kodu i danych nie wystarczy, jeśli zmieniły się biblioteki, parametry lub losowość treningu. Wersjonuj także opis środowiska i konfigurację eksperymentu.
- **Uprawnienia i sekrety:** adres magazynu można zapisać w `.dvc/config`, ale tokeny i hasła powinny być poza Git, np. w lokalnej konfiguracji `.dvc/config.local` lub mechanizmie poświadczeń dostawcy chmury. Historia wskaźników w Git nie nadaje automatycznie dostępu do danych.
- **Usuwanie obiektów:** nie czyść magazynu bez sprawdzenia, które wersje mają pozostać odtwarzalne. `dvc gc` służy do usuwania niepotrzebnych obiektów; jego niewłaściwe użycie, szczególnie wobec współdzielonego magazynu, może uniemożliwić odtworzenie starszych eksperymentów.

### Wracamy do ćwiczenia

**Zadanie:** zainicjuj DVC, dodaj plik danych pod kontrolę, skonfiguruj magazyn lokalny i wyślij do niego dane.

### Zadanie w terminalu

Najpierw wykonaj wcześniejsze komórki notebooka, aby powstały plik danych i manifest. Otwórz terminal w **katalogu głównym repozytorium zajęć**. Wybranie kernela w notebooku nie gwarantuje aktywacji środowiska w osobnym terminalu.

### Aktywacja środowiska (macOS / Linux, bash lub zsh)

DVC jest zainstalowane w środowisku wirtualnym workspace'a. W terminalu trzeba aktywować to środowisko albo wskazać bezpośrednio jego interpreter. Dla środowiska `.venv` użyj:

```bash
source .venv/bin/activate
python -c 'import sys; print(sys.executable)'
python -m dvc --version
```

- `source .venv/bin/activate` wykonuje skrypt aktywacyjny w bieżącej powłoce i dodaje katalog programów środowiska na początek `PATH`. Od tej chwili `python` i `dvc` powinny pochodzić z tego środowiska.
- `python -c 'import sys; print(sys.executable)'` uruchamia krótki kod wypisujący ścieżkę interpretera. Porównaj ją ze środowiskiem wybranym dla notebooka.
- `python -m dvc --version` uruchamia moduł DVC przez ten interpreter i wypisuje wersję. Błąd `No module named dvc` oznacza, że wybrane środowisko nie ma DVC — sprawdź wybór środowiska, zanim przejdziesz dalej.

Jeśli widzisz już prefiks `(.venv)` i ścieżka interpretera jest poprawna, nie musisz aktywować środowiska ponownie. Aktywacja dotyczy tylko bieżącego terminala i pozostaje ważna po zmianie katalogu przez `cd`. Dalsze przykłady używają krótszej formy `dvc`; możesz ją zastąpić `python -m dvc`, aby jawnie wybrać DVC z aktywnego interpretera.

Obejmij dane kontrolą wersji: zainicjuj DVC, dodaj `data/iris.csv`, skonfiguruj magazyn `../magazyn` i wyślij do niego dane. Poniżej znajdziesz polecenia do wykonania samodzielnie; nie uruchamiaj ich jako kodu notebooka.

### Polecenia do wykonania w terminalu

Przejdź z katalogu głównego repozytorium do katalogu projektu ćwiczeniowego:

```bash
cd artifacts/zajecia_03/projekt
pwd
```

- `cd artifacts/zajecia_03/projekt` zmienia katalog roboczy terminala. Ścieżka jest względna wobec katalogu głównego repozytorium, więc wykonaj to przejście tylko raz. `cd` nie zmienia katalogu roboczego kernela notebooka.
- `pwd` wypisuje pełną ścieżkę bieżącego katalogu. Dzięki temu sprawdzasz, na którym projekcie będą działały następne polecenia.

Sprawdź, czy jesteś w `artifacts/zajecia_03/projekt`. Wykonuj kolejne polecenia pojedynczo i przechodź dalej tylko wtedy, gdy poprzednie zakończyło się powodzeniem:

```bash
dvc init --no-scm
dvc add data/iris.csv
dvc remote add -d magazyn ../magazyn
dvc push
```

| Polecenie | Co dokładnie robi |
|---|---|
| `dvc init --no-scm` | Tworzy konfigurację projektu w `.dvc` i plik `.dvcignore`. Opcja `--no-scm` pozwala pracować bez integracji z Git; dotyczy tylko tego izolowanego ćwiczenia. Nie wysyła danych i nie tworzy commita. |
| `dvc add data/iris.csv` | Oblicza skrót pliku, zapisuje jego zawartość w lokalnym cache DVC i tworzy wskaźnik `data/iris.csv.dvc` z metadanymi potrzebnymi do odtworzenia danych. Nie wysyła jeszcze danych do magazynu. |
| `dvc remote add -d magazyn ../magazyn` | Zapisuje w `.dvc/config` magazyn o nazwie `magazyn`. `-d` ustawia go jako domyślny dla `push` i `pull`. `..` oznacza katalog nadrzędny: tutaj magazynem jest `artifacts/zajecia_03/magazyn`, poza katalogiem `projekt`. Słowo „remote” nie wymaga serwera — na zajęciach to lokalny katalog. |
| `dvc push` | Kopiuje brakujące obiekty z lokalnego cache do domyślnego magazynu. Dopiero powodzenie tej operacji pozwala bezpiecznie przejść do symulacji utraty lokalnych danych. Nie wysyła kodu ani commitów Git. |

Przy ponownym wykonywaniu ćwiczenia pomiń `dvc init --no-scm`, jeśli katalog `.dvc` już istnieje. Sprawdź `dvc remote list`: jeśli `magazyn` już wskazuje `../magazyn`, pomiń również `dvc remote add`. Jeśli wskazuje inne miejsce, wyjaśnij różnicę przed kontynuowaniem — nie nadpisuj konfiguracji w ciemno.

`dvc remote list` tylko wyświetla nazwy i adresy skonfigurowanych magazynów; niczego nie przesyła ani nie zmienia. Adres może być pokazany jako ścieżka bezwzględna — sprawdź, czy prowadzi do tego samego katalogu `artifacts/zajecia_03/magazyn`.

Obejrzyj wskaźnik i porównaj rozmiary plików:

```bash
cat data/iris.csv.dvc
wc -c data/iris.csv.dvc data/iris.csv
```

- `cat data/iris.csv.dvc` wypisuje tekst wskaźnika: znajdziesz tam m.in. skrót, rozmiar danych i ścieżkę pliku. Nie wypisuje całego zbioru danych.
- `wc -c ...` liczy bajty każdego wskazanego pliku (`-c`) i pokazuje również sumę. Porównujesz rozmiar wskaźnika z rozmiarem CSV; to nie jest sprawdzenie sumy kontrolnej.

To właśnie mały wskaźnik powinien trafić do Git w rzeczywistym projekcie — nie dane. W tym ćwiczeniu z `--no-scm` nie tworzymy historii Git; pełny wariant znajdziesz w sekcji 13. Pozostaw terminal w tym katalogu na następny krok.

## 7. Punkt kontrolny 2 — utrata pliku danych

W terminalu usuniesz plik danych **oraz lokalną pamięć podręczną**, a następnie odtworzysz dane z magazynu. Zostanie tylko wskaźnik i magazyn — tak jak przy pobieraniu danych po sklonowaniu repozytorium. To symulacja utraty lokalnych danych, nie test świeżego klonu Git.

**Do opisania:** czy odtworzony plik jest tym samym plikiem, czy tylko podobnym? Skąd to wiesz?

### Odtworzenie danych — terminal

**Wykonaj ten krok dopiero po udanym `dvc push`.** Poniższe polecenia usuwają wyłącznie dane i pamięć podręczną projektu ćwiczeniowego. Nie usuwaj manifestu, wskaźnika `.dvc` ani katalogu `../magazyn`.

W terminalu pozostawionym w `artifacts/zajecia_03/projekt` sprawdź katalog roboczy i lokalizację pamięci podręcznej:

```bash
pwd
dvc cache dir
```

`pwd` potwierdza katalog roboczy, a `dvc cache dir` wypisuje lokalizację lokalnego cache — nie magazynu `../magazyn`. Usunięcie cache wymusza późniejsze pobranie danych z magazynu zamiast ich odtworzenia z lokalnej kopii.

Jeżeli otwierasz nowy terminal, najpierw aktywuj środowisko z katalogu głównego repozytorium, jak w sekcji 6, a następnie wykonaj `cd artifacts/zajecia_03/projekt`. Kontynuuj tylko wtedy, gdy `dvc cache dir` wskazuje `.dvc/cache` **wewnątrz tego projektu**; przy innej lokalizacji nie usuwaj pamięci podręcznej i wyjaśnij konfigurację.

Wykonuj polecenia pojedynczo, sprawdzając ich wynik:

```bash
rm data/iris.csv
rm -rf .dvc/cache
```

- `rm data/iris.csv` usuwa roboczy plik danych, ale pozostawia manifest i wskaźnik. Jeśli plik jest chroniony przed zapisem i pojawi się pytanie o usunięcie, potwierdź `y` tylko po sprawdzeniu tej konkretnej ścieżki. Jeśli plik został już usunięty przy poprzedniej próbie, pomiń ten krok.
- `rm -rf .dvc/cache` usuwa katalog lokalnego cache wraz z zawartością. `-r` oznacza usuwanie rekurencyjne, a `-f` wyłącza pytania o pliki chronione przed zapisem oraz ignoruje brak wskazanej ścieżki. **To operacja nieodwracalna: użyj dokładnie tej ścieżki, wyłącznie w sprawdzonym katalogu projektu i po udanym `dvc push`. Nie używaj `sudo`.**

Przed odtworzeniem sprawdź poleceniami `ls data` oraz `ls .dvc`, że nie ma już `iris.csv` ani katalogu `cache`, ale pozostały manifest i wskaźnik. Następnie pobierz dane z magazynu:

```bash
dvc pull
ls -l data/iris.csv
```

- `ls data` i `ls .dvc` wyświetlają nazwy plików w podanych katalogach, bez ich modyfikowania. Po usunięciu CSV nadal powinny być widoczne `iris.manifest.json` i `iris.csv.dvc` w `data`.
- `dvc pull` pobiera brakujące obiekty z domyślnego magazynu do cache, a następnie odtwarza plik roboczy zgodnie ze wskaźnikiem `data/iris.csv.dvc`. Nie pobiera ponownie CSV z pierwotnego adresu HTTP.
- `ls -l data/iris.csv` pokazuje szczegóły odtworzonego pliku, m.in. uprawnienia i rozmiar. Potwierdza istnienie pliku, ale dopiero poniższe porównanie SHA-256 z zachowanym manifestem sprawdza jego tożsamość.

Po udanym odtworzeniu wróć do notebooka i uruchom poniższą komórkę. **Nie generuj manifestu od nowa** — porównanie ma dotyczyć wersji zapisanej przed usunięciem danych.

In [ ]:
verify_dataset(DATA_PATH, MANIFEST_PATH)

print("\nSuma kontrolna zgadza się z manifestem sprzed usunięcia —")
print("to jest ten sam plik co do bajtu, a nie jego rekonstrukcja.")

## 7a. Po co odzyskiwać dane? Odtworzenie wyniku eksperymentu

Samo odzyskanie CSV pokazuje tylko mechanizm. Teraz użyjemy danych do **treningu modelu rozpoznającego gatunek irysa**, porównamy dwie wersje danych i odtworzymy pierwszy wynik w świeżym klonie. Kod treningu czyta zwykły plik CSV — to DVC dostarcza pod tę ścieżkę właściwą wersję zawartości.

**Scenariusz:** na poprawnie oznaczonych danych A model działa dobrze. Następnie otrzymujemy eksport B z błędnymi etykietami treningowymi. Kod i zbiór testowy pozostają bez zmian, ale wynik się pogarsza. Zamiast zgadywać, co było w poprzednim CSV, wrócimy do wersji A zapisanej przez Git i DVC.

**Co przechowujemy i po co?**

| Element | Gdzie go zapisujemy? | Do czego posłuży? |
|---|---|---|
| `train.py` | Git | Odtworzenie tej samej metody treningu i ustawień |
| `data/train.csv` w wersjach A i B | Cache i osobny magazyn DVC | Odzyskanie dokładnych przykładów oraz etykiet użytych w danym treningu |
| Niezmienny `data/test.csv` | Cache i magazyn DVC | Uczciwe porównanie obu modeli na tych samych danych |
| Wskaźniki `data/*.csv.dvc` | Git | Powiązanie commita z konkretną zawartością danych |
| `metrics.json` | Git | Zachowanie trafności i SHA-256 obu plików, aby sprawdzić odtworzenie wyniku |

W tym krótkim przykładzie model trenuje się szybko, więc nie zapisujemy jego wag. W większym projekcie można również zachować model jako artefakt DVC, aby nie trenować go ponownie.

### Przygotowanie — notebook

Poniższa komórka jest gotowa. Uruchom ją **raz**, po odtworzeniu i weryfikacji Iris w poprzedniej sekcji. Tworzy osobny katalog `artifacts/zajecia_03/eksperyment_dvc`, dwa rozłączne zbiory i skrypt treningu. Nie zmienia `DATA_PATH` ani manifestu wcześniejszego ćwiczenia. Dalsze sekcje o kontrakcie danych nadal korzystają z oryginalnego Iris.

Podział jest stratyfikowany (zachowuje proporcje gatunków) i ma ustalone `random_state=42`. Model też ma ustaloną losowość. To pozwala badać zmianę danych, a nie zmianę podziału lub ustawień.

In [ ]:
from sklearn.model_selection import train_test_split

verify_dataset(DATA_PATH, MANIFEST_PATH)
EXPERIMENT_DIR = BASE / "eksperyment_dvc"
if EXPERIMENT_DIR.exists():
    raise FileExistsError(
        f"{EXPERIMENT_DIR} już istnieje. Nie nadpisujemy eksperymentu. "
        "Jeśli przygotowanie zakończyło się wcześniej powodzeniem, przejdź do następnego kroku."
    )

experiment_frame = pd.read_csv(DATA_PATH)
train_a, test_fixed = train_test_split(
    experiment_frame, test_size=0.3, random_state=42,
    stratify=experiment_frame["species"],
)
(EXPERIMENT_DIR / "data").mkdir(parents=True)
train_a.to_csv(EXPERIMENT_DIR / "data" / "train.csv", index=False)
test_fixed.to_csv(EXPERIMENT_DIR / "data" / "test.csv", index=False)

write_file(
    EXPERIMENT_DIR / "train.py",
    '''
    import argparse
    import hashlib
    import json
    from pathlib import Path

    import pandas as pd
    import sklearn
    from sklearn.metrics import accuracy_score
    from sklearn.tree import DecisionTreeClassifier

    root = Path(__file__).resolve().parent
    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default="metrics.json")
    output_path = root / parser.parse_args().output
    train_path = root / "data" / "train.csv"
    test_path = root / "data" / "test.csv"
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(train[features], train["species"])
    predictions = model.predict(test[features])
    metrics = {
        "accuracy": float(accuracy_score(test["species"], predictions)),
        "train_sha256": hashlib.sha256(train_path.read_bytes()).hexdigest(),
        "test_sha256": hashlib.sha256(test_path.read_bytes()).hexdigest(),
        "sklearn_version": sklearn.__version__,
    }
    result = json.dumps(metrics, indent=2) + "\\n"
    output_path.write_text(result, encoding="utf-8")
    print(result)
    ''',
)
print(f"Projekt eksperymentu: {EXPERIMENT_DIR.resolve()}")
print(f"Trening: {len(train_a)} wierszy; test: {len(test_fixed)} wierszy")

### Zadanie 1 — zapisz wynik dla danych A (terminal)

W terminalu z aktywnym środowiskiem notebooka przejdź **z katalogu głównego repozytorium zajęć** do nowego projektu. Jeśli terminal został w poprzednim katalogu `projekt`, najpierw wróć do katalogu głównego repozytorium. Wykonuj polecenia pojedynczo i zatrzymaj się przy błędzie:

```bash
cd artifacts/zajecia_03/eksperyment_dvc
pwd
git init -b main
dvc init
dvc add data/train.csv data/test.csv
dvc remote add -d magazyn ../eksperyment_dvc_magazyn
dvc push
python train.py
git add .dvc .dvcignore data/.gitignore data/train.csv.dvc data/test.csv.dvc train.py metrics.json
git status
git commit -m "Eksperyment A: poprawne etykiety"
git tag wersja-a
```

- `pwd` pozwala sprawdzić, że jesteś w **nowym projekcie**, zanim wykonasz `git init -b main`. To polecenie celowo tworzy osobne, lokalne repozytorium z gałęzią `main`. Katalog `artifacts` jest ignorowany przez repozytorium zajęć, więc nie dodawaj tego eksperymentu do jego historii. W tym scenariuszu ćwiczymy już pełną integrację, dlatego `dvc init` nie ma `--no-scm`.
- `dvc add` tworzy dwa wskaźniki, a `dvc push` zapisuje obie zawartości w osobnym magazynie obok eksperymentu. Nie korzystamy z magazynu poprzedniego ćwiczenia.
- `python train.py` faktycznie uczy model na `data/train.csv` i mierzy trafność (`accuracy`, udział poprawnych przewidywań) na `data/test.csv`. Zapisuje wynik i identyfikatory danych w `metrics.json`.
- `git add` i `git commit` zapisują razem kod, wskaźniki i wynik; `git status` pozwala sprawdzić przygotowane pliki. Jeśli Git zgłasza brak tożsamości autora, skonfiguruj własne `user.name` i `user.email` lokalnie w tym repozytorium i ponów commit.
- `git tag wersja-a` nadaje bieżącemu commitowi łatwą do zapamiętania nazwę. Dzięki niej później wybierzemy dokładnie ten eksperyment. Nie nadpisuj istniejącego tagu przy ponownym wykonywaniu ćwiczenia.

Nie wykonujemy `git push`: to lokalna demonstracja, bez zdalnego serwera Git. Odczytaj trafność z terminala, a następnie wróć do notebooka.

### Zadanie 2 — nowy eksport z błędnymi etykietami (notebook)

Uruchom poniższą komórkę **raz, po zapisaniu i oznaczeniu wersji A w Git**. Zachowa jej wynik w pamięci notebooka, a następnie utworzy wersję B, cyklicznie zmieniając wszystkie etykiety treningowe: `setosa → versicolor → virginica → setosa`. To celowy, skrajny błąd eksportu, a nie realistyczna symulacja niewielkiego szumu.

Nie zmieniamy pomiarów roślin, kodu modelu ani danych testowych. **Przewidź wynik:** czy kontrakt sprawdzający tylko dozwolone nazwy gatunków wykryje tę podmianę? Czy poprawny schemat gwarantuje poprawność etykiet?

In [ ]:
experiment_a = json.loads((EXPERIMENT_DIR / "metrics.json").read_text(encoding="utf-8"))
experiment_train_path = EXPERIMENT_DIR / "data" / "train.csv"
experiment_test_path = EXPERIMENT_DIR / "data" / "test.csv"
if sha256_of(experiment_train_path) != experiment_a["train_sha256"]:
    raise RuntimeError("Dane nie odpowiadają zapisanemu wynikowi A. Nie zmieniamy etykiet ponownie.")
if sha256_of(experiment_test_path) != experiment_a["test_sha256"]:
    raise RuntimeError("Zbiór testowy został zmieniony. Przywróć wersję A przed kontynuowaniem.")

train_b = pd.read_csv(experiment_train_path)
label_rotation = {"setosa": "versicolor", "versicolor": "virginica", "virginica": "setosa"}
if not train_b["species"].isin(label_rotation).all():
    raise ValueError("Nieznana etykieta gatunku — sprawdź dane treningowe.")
train_b["species"] = train_b["species"].map(label_rotation)
train_b.to_csv(experiment_train_path, index=False)
print("Zmieniono tylko etykiety treningowe. Teraz zapisz i uruchom eksperyment B w terminalu.")

### Zadanie 3 — zapisz i porównaj wynik B (terminal, następnie notebook)

W tym samym terminalu, nadal w `eksperyment_dvc`, wykonaj:

```bash
dvc status
dvc add data/train.csv
dvc push
python train.py
git add data/train.csv.dvc metrics.json
git status
git commit -m "Eksperyment B: błędne etykiety treningowe"
git tag wersja-b
```

`dvc status` powinno wykazać zmianę danych treningowych. `dvc add` aktualizuje ich wskaźnik, a `dvc push` zachowuje nową zawartość **obok** poprzedniej. Nie dodajemy ponownie testu, bo się nie zmienił. `python train.py` uruchamia ten sam kod, tym razem na B. Commit i tag zapisują drugi eksperyment; w historii są teraz dwa wyniki i dwa wskaźniki treningowe.

Po powodzeniu wszystkich poleceń uruchom poniższą komórkę porównującą wyniki. Nie uruchamiaj ponownie komórki zmieniającej etykiety.

In [ ]:
experiment_b = json.loads((EXPERIMENT_DIR / "metrics.json").read_text(encoding="utf-8"))
if experiment_a["test_sha256"] != experiment_b["test_sha256"]:
    raise RuntimeError("Porównanie nieuczciwe: zmienił się zbiór testowy.")
if experiment_a["train_sha256"] == experiment_b["train_sha256"]:
    raise RuntimeError("Brak nowej wersji danych. Wykonaj trening B w terminalu.")
if experiment_b["accuracy"] >= experiment_a["accuracy"]:
    raise RuntimeError("Nie zaobserwowano oczekiwanego pogorszenia. Sprawdź kolejność kroków i dane.")
display(pd.DataFrame([experiment_a, experiment_b], index=["A — poprawne etykiety", "B — błędne etykiety"]))
print(f"Spadek trafności: {experiment_a['accuracy'] - experiment_b['accuracy']:.3f}")

### Zadanie 4 — odtwórz pierwszy wynik w świeżym klonie (terminal)

Teraz odgrywasz rolę drugiej osoby, która chce sprawdzić eksperyment A, mimo że bieżąca wersja projektu to B. **Nie kopiuj ręcznie CSV.** Z terminala w `eksperyment_dvc` wykonaj:

```bash
git status --short
cd ..
git clone eksperyment_dvc eksperyment_dvc_odtworzenie
cd eksperyment_dvc_odtworzenie
git switch --detach wersja-a
dvc pull
python train.py --output metrics-recomputed.json
git diff --no-index --exit-code -- metrics.json metrics-recomputed.json
```

- Przed klonowaniem `git status --short` powinno niczego nie wypisać: oba eksperymenty mają być zapisane w commitach. Przy niezapisanych zmianach zatrzymaj się i dokończ poprzedni krok.
- `cd ..` przechodzi do `artifacts/zajecia_03`. `git clone` tworzy obok nową kopię lokalnego repozytorium; nie wymaga GitHuba. Użyj nowego katalogu — jeśli `eksperyment_dvc_odtworzenie` już istnieje, nie nadpisuj go i nie usuwaj jego zawartości w ciemno.
- Git pobiera kod, wyniki i wskaźniki, **ale nie ignorowane CSV ani cache DVC**. `git switch --detach wersja-a` wybiera kod i wskaźniki pierwszego eksperymentu bez zmieniania gałęzi `main` w źródłowym projekcie.
- `dvc pull` musi pobrać zawartość wskazaną przez A z magazynu. Względny adres `../eksperyment_dvc_magazyn` działa również z tego klonu, bo leży on obok oryginału. Na innym komputerze potrzebny byłby wspólny, dostępny magazyn.
- `python train.py --output metrics-recomputed.json` naprawdę powtarza obliczenie na odzyskanych danych — nie tylko odczytuje historyczną metrykę. Opcja `--output` zapisuje nowy wynik w osobnym pliku, aby nie nadpisać historycznego `metrics.json` pobranego z Git. Bez wykonania treningu ten nowy plik nie powstanie.
- `git diff --no-index --exit-code -- metrics.json metrics-recomputed.json` porównuje dwa pliki na dysku (`--no-index`), niezależnie od ich śledzenia przez Git. Brak różnic i kod zakończenia 0 oznaczają zgodność. Kod 1 oznacza różnicę, którą trzeba zbadać, nie sukces.

Środowisko pozostaje aktywne po `cd`; w nowym terminalu aktywuj je przed uruchomieniem DVC i treningu. Następnie wróć do notebooka i sprawdź odtworzenie:

In [ ]:
restored_experiment_dir = BASE / "eksperyment_dvc_odtworzenie"
experiment_restored = json.loads(
    (restored_experiment_dir / "metrics-recomputed.json").read_text(encoding="utf-8")
)
for split in ("train", "test"):
    actual_hash = sha256_of(restored_experiment_dir / "data" / f"{split}.csv")
    if actual_hash != experiment_a[f"{split}_sha256"]:
        raise RuntimeError(f"Odtworzony zbiór {split} nie odpowiada wersji A.")
if experiment_restored != experiment_a:
    raise RuntimeError("Nie odtworzono wyniku A. Sprawdź dane, kod i wersję środowiska.")
display(pd.DataFrame(
    [experiment_a, experiment_b, experiment_restored],
    index=["A", "B", "A — odtworzony eksperyment"],
))
print("Odzyskano dane A i odtworzono wynik A. Oryginalny projekt eksperymentu nadal przechowuje wersję B.")

**To jest praktyczny cel magazynu:** nie tylko ratowanie przypadkowo usuniętego pliku, ale umożliwienie ponownego wykonania konkretnego eksperymentu i wyjaśnienia zmiany wyniku. W tym przykładzie celowo zmienialiśmy wyłącznie dane; w rzeczywistym projekcie trzeba kontrolować również kod, parametry i środowisko. Wyniki liczbowe zależą od użytej wersji Iris i bibliotek, dlatego nie przepisujemy stałej oczekiwanej trafności — porównujemy faktycznie wykonane przebiegi.

Przejdź teraz do kontraktu danych. Zmienne `DATA_PATH` i `MANIFEST_PATH` nadal wskazują oryginalny zbiór wcześniejszego ćwiczenia, nie błędnie oznaczony eksport B.

## 8. Kontrakt danych

Suma kontrolna sprawdza tożsamość pliku. Kontrakt sprawdza jego **sens**: czy wartości mieszczą się w dopuszczalnych zakresach, czy nie ma braków tam, gdzie ich być nie może, czy zbiór kategorii jest zamknięty.

Różnica jest zasadnicza: nowa dostawa danych **ma** mieć inną sumę kontrolną, ale **musi** spełniać ten sam kontrakt.

**Zadanie:** zdefiniuj schemat `pandera` dla zbioru. Uwzględnij typy, zakresy wartości, zamknięty zbiór gatunków, brak wartości pustych i tryb ścisły (żadnych dodatkowych kolumn).

In [ ]:
# Zadanie: zdefiniuj kontrakt danych.


In [ ]:
%%skip True
SPECIES = ["setosa", "versicolor", "virginica"]

SCHEMA = pa.DataFrameSchema(
    columns={
        "sepal_length": pa.Column(float, pa.Check.in_range(3.0, 9.0), nullable=False),
        "sepal_width": pa.Column(float, pa.Check.in_range(1.0, 5.0), nullable=False),
        "petal_length": pa.Column(float, pa.Check.in_range(0.5, 8.0), nullable=False),
        "petal_width": pa.Column(float, pa.Check.in_range(0.05, 3.0), nullable=False),
        "species": pa.Column(str, pa.Check.isin(SPECIES), nullable=False),
    },
    checks=[
        # Regula miedzykolumnowa: platek nigdy nie jest dluzszy od dzialki kielicha.
        pa.Check(lambda frame: frame["petal_length"] <= frame["sepal_length"], name="platek_krotszy"),
    ],
    strict=True,
    coerce=False,
)

print("Kontrakt obejmuje:")
for name, column in SCHEMA.columns.items():
    rules = ", ".join(str(check) for check in column.checks) or "tylko typ"
    print(f"  {name:14s} {column.dtype}  {rules}")
print(f"  + {len(SCHEMA.checks)} reguła międzykolumnowa")
print(f"  + tryb ścisły: {SCHEMA.strict}")

## 9. Walidacja i raport

Walidację uruchamiamy w trybie **leniwym** (`lazy=True`). W bibliotece Pandera oznacza to, że walidator **nie przerywa sprawdzania po pierwszym błędzie**, lecz zbiera wykryte naruszenia i na końcu zgłasza je razem w wyjątku `pa.errors.SchemaErrors`. Szczegóły są dostępne w `errors.failure_cases`, z którego poniższy kod tworzy raport. Bez tego ustawienia (`lazy=False`, domyślnie) walidacja zatrzymałaby się na pierwszym błędzie.

Na przykład: jeśli w danych są jednocześnie brakujące wartości i wartości spoza dozwolonego zakresu, tryb leniwy pozwala zobaczyć oba rodzaje problemów w jednym przebiegu zamiast poprawiać je po kolei i za każdym razem ponawiać walidację. „Leniwy” oznacza tutaj **odroczenie zgłoszenia błędów do końca sprawdzania**, a nie odłożenie samej walidacji na później ani pomijanie reguł.

Wynik zapisujemy jako raport — to on trafi później do artefaktów przebiegu CI.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
def validate(frame: pd.DataFrame, label: str) -> dict:
    try:
        SCHEMA.validate(frame, lazy=True)
        report = {"dataset": label, "valid": True, "violations": []}
    except pa.errors.SchemaErrors as errors:
        cases = errors.failure_cases
        report = {
            "dataset": label,
            "valid": False,
            "violations": (
                cases.groupby(["column", "check"], dropna=False)
                .size()
                .reset_index(name="count")
                .to_dict(orient="records")
            ),
        }
    (REPORTS_DIR / f"validation_{label}.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
    )
    return report


raw = pd.read_csv(DATA_PATH)
result = validate(raw, "produkcyjny")
print(f"Zbiór zgodny z kontraktem: {result['valid']}")

## 10. Punkt kontrolny 3 — złamany kontrakt

Symulujemy typową dostawę danych po zmianie w systemie źródłowym: kilka braków, jedna wartość poza zakresem, nowa kategoria i dodatkowa kolumna.

**Do opisania:** które z tych naruszeń wykryłby zwykły `read_csv` bez kontraktu? Które przeszłyby niezauważone aż do wyników modelu?

In [ ]:
broken = raw.copy()
broken.loc[0:4, "sepal_width"] = None
broken.loc[10, "petal_length"] = 87.0
broken.loc[11, "species"] = "Setosa"
broken["zrodlo_rekordu"] = "system_b"

report = validate(broken, "uszkodzony")
print(f"Zbiór zgodny z kontraktem: {report['valid']}\n")
display(pd.DataFrame(report["violations"]))

print(
    "Największe ryzyko niesie 'Setosa' zapisana wielką literą:\n"
    "nie jest brakiem ani wartością odstającą, więc przejdzie przez większość kontroli,\n"
    "a w modelu utworzy nową, prawie pustą klasę."
)

## 11. Wyciek etykiety

Najgroźniejsze dane to te, które wyglądają na świetne. Kolumna silnie skorelowana z celem — bo powstała **po** nim albo **z** niego — daje znakomity wynik w ocenie i bezużyteczny model w produkcji.

**Zadanie:** dopisz do zbioru kolumnę zależną od etykiety, porównaj jakość modelu z nią i bez niej, a następnie napisz `find_leaks(frame, target)` wskazującą podejrzane kolumny.

In [ ]:
# Zadanie: wykryj wyciek etykiety.


In [ ]:
%%skip True
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
rng = np.random.default_rng(42)

leaky = raw.copy()
codes = leaky["species"].astype("category").cat.codes
# Kolumna, ktora w rzeczywistosci powstaje PO poznaniu etykiety,
# na przyklad numer kolejny nadawany przy katalogowaniu okazu.
leaky["numer_katalogowy"] = codes.astype("int64") * 1000 + rng.integers(0, 999, size=len(leaky))

scores = {}
for label, columns in (("bez wycieku", FEATURES), ("z wyciekiem", FEATURES + ["numer_katalogowy"])):
    scores[label] = float(
        cross_val_score(
            RandomForestClassifier(n_estimators=200, random_state=42),
            leaky[columns], leaky["species"], cv=5, scoring="accuracy",
        ).mean()
    )
    print(f"{label:14s} trafność {scores[label]:.4f}")


def find_leaks(frame: pd.DataFrame, target: str, threshold: float = 0.9) -> pd.DataFrame:
    """Kolumna, ktora sama w sobie prawie idealnie przewiduje cel, jest podejrzana."""
    labels = frame[target].astype("category").cat.codes
    rows = []
    for column in frame.columns:
        if column == target or not pd.api.types.is_numeric_dtype(frame[column]):
            continue
        alone = float(
            cross_val_score(
                RandomForestClassifier(n_estimators=100, random_state=42),
                frame[[column]], labels, cv=5, scoring="accuracy",
            ).mean()
        )
        rows.append({"kolumna": column, "trafnosc_sama": round(alone, 4), "podejrzana": alone >= threshold})
    return pd.DataFrame(rows).sort_values("trafnosc_sama", ascending=False)


print()
display(find_leaks(leaky, "species"))
print(
    "Sam wysoki wynik nie jest dowodem wycieku — o tym rozstrzyga pytanie,\n"
    "czy ta kolumna będzie znana w momencie predykcji. Tu nie będzie."
)

## 12. Metryczka zbioru

Metryczka (*dataset card*) to dokument, który odpowiada na pytania zadawane przez osoby, których nie było przy powstawaniu zbioru: skąd te dane, kto za nie odpowiada, co wolno z nimi zrobić, jakie mają ograniczenia.

To jest element **governance**, a nie dokumentacji technicznej. Jego brak ujawnia się dopiero wtedy, gdy ktoś pyta, czy wolno wytrenować na tych danych model komercyjny.

**Zadanie:** wygeneruj `dataset_card.md` na podstawie manifestu i kontraktu. Uwzględnij pochodzenie, licencję, właściciela, zakres dopuszczalnego użycia, znane ograniczenia i kategorie danych osobowych.

In [ ]:
# Zadanie: wygeneruj metryczkę zbioru.


In [ ]:
%%skip True
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
columns_table = "\n".join(
    f"| `{name}` | {dtype} | {'cel' if name == 'species' else 'cecha'} |"
    for name, dtype in manifest["columns"].items()
)

write_file(
    REPORTS_DIR / "dataset_card.md",
    f"""
    # Metryczka zbioru — {manifest['name']}

    ## Tożsamość

    | Pole | Wartość |
    |---|---|
    | Źródło | {manifest['source']} |
    | Pobrano | {manifest['retrieved_at']} |
    | Suma kontrolna (SHA-256) | `{manifest['sha256']}` |
    | Rozmiar | {manifest['size_bytes']} B |
    | Liczba wierszy | {manifest['rows']} |

    ## Zawartość

    | Kolumna | Typ | Rola |
    |---|---|---|
{textwrap.indent(columns_table, '    ')}

    ## Pochodzenie i prawa

    - **Właściciel merytoryczny:** prowadzący przedmiot (zbiór dydaktyczny).
    - **Licencja:** zbiór publiczny, powszechnie stosowany w dydaktyce; przed użyciem
      komercyjnym należy sprawdzić warunki u dysponenta źródła.
    - **Dane osobowe:** brak. Zbiór opisuje pomiary roślin, nie osoby.
    - **Dopuszczalne użycie:** dydaktyka, testy potoku, przykłady w dokumentacji.

    ## Znane ograniczenia

    - Zbiór jest **mały i zrównoważony**, co nie odpowiada warunkom produkcyjnym;
      wyniki uzyskane na nim nie przenoszą się na zadania z klasami rzadkimi.
    - Pochodzi z jednego pomiaru w jednym miejscu — nie zawiera zmienności
      w czasie, więc **nie da się na nim badać dryfu** bez sztucznego wprowadzenia zmian.
    - Trafność powyżej 0.95 jest tu normą, nie osiągnięciem. Wysoki wynik na tym
      zbiorze nie świadczy o jakości rozwiązania.

    ## Kontrakt

    Zbiór musi spełniać schemat zdefiniowany w notebooku zajęć 3:
    typy kolumn, zakresy wartości, zamknięty zbiór gatunków, brak wartości pustych,
    brak kolumn dodatkowych oraz reguła `petal_length <= sepal_length`.

    ## Wersjonowanie

    Plik jest pod kontrolą DVC. W repozytorium znajduje się wskaźnik
    `data/iris.csv.dvc`; zawartość przechowuje magazyn skonfigurowany jako `magazyn`.
    Odtworzenie: `dvc pull`.
    """,
)

print()
print((REPORTS_DIR / "dataset_card.md").read_text(encoding="utf-8")[:900])

## 13. Poza notebookiem — dane w repozytorium

W projekcie zaliczeniowym DVC uruchamiasz **wewnątrz repozytorium Git**, bez `--no-scm`. Wtedy plik `.dvc` jest wersjonowany razem z kodem i historia danych staje się częścią historii projektu.

To osobny scenariusz dla własnego projektu, nie dalszy ciąg poleceń w katalogu ćwiczeniowym. Otwórz terminal w katalogu głównym własnego repozytorium Git i aktywuj jego środowisko z DVC, analogicznie do sekcji 6. Przykład zakłada, że masz już plik `data/surowe.csv`; zastąp tę nazwę rzeczywistą nazwą swoich danych.

### Krok 1 — inicjalizacja

```bash
dvc init
git add .dvc .dvcignore
git commit -m "Wersjonowanie danych przez DVC"
```

- `dvc init` inicjalizuje DVC z integracją Git; wykonujesz je raz w danym projekcie.
- `git add .dvc .dvcignore` dodaje konfigurację DVC i reguły ignorowania DVC do obszaru przygotowania Git (staging). `.dvcignore` nie zastępuje `.gitignore`: określa pliki pomijane przez DVC, nie przez Git.
- `git commit -m "..."` zapisuje przygotowane zmiany w lokalnej historii Git z podanym opisem (`-m`). Nie wysyła ich na serwer. Commit obejmuje wszystkie przygotowane zmiany, więc przed nim sprawdź `git status` — pokazuje stan repozytorium i pliki przygotowane do commita.

### Krok 2 — objęcie danych kontrolą

```bash
dvc add data/surowe.csv
git add data/surowe.csv.dvc data/.gitignore
git commit -m "Dane wejściowe v1"
```

`dvc add` tworzy lokalną kopię danych w cache i wskaźnik, tak jak w ćwiczeniu. W trybie z integracją Git **sam dopisuje** plik danych do `.gitignore`. `git add data/surowe.csv.dvc data/.gitignore` przygotowuje do commita wskaźnik i regułę ignorowania CSV, nie sam CSV. Kolejny `git commit` utrwala tę wersję wskaźnika w historii Git.

### Krok 3 — magazyn zdalny

Na zajęciach wystarczy katalog:

```bash
dvc remote add -d magazyn ../dane-magazyn
dvc push
```

`dvc remote add` zapisuje nazwę i adres magazynu, a `-d` ustawia go jako domyślny. Tutaj `../dane-magazyn` to katalog obok repozytorium projektu. `dvc push` wysyła do niego zawartość cache, nie kod. Po skonfigurowaniu magazynu wykonaj `git add .dvc/config`, a następnie `git commit -m "Konfiguracja magazynu danych"`: dzięki temu kolejny klon otrzyma również adres magazynu.

W projekcie zespołowym w tym miejscu podaje się adres magazynu obiektowego (S3, Azure Blob, Google Cloud Storage) albo serwera SSH. Konfiguracja jest wersjonowana, **poświadczenia nie** — te trafiają do `.dvc/config.local`, który jest ignorowany przez Git.

### Krok 4 — sprawdzenie z drugiej strony

```bash
git clone ADRES_REPOZYTORIUM sprawdzenie
cd sprawdzenie
dvc pull
```

Zastąp `ADRES_REPOZYTORIUM` rzeczywistym adresem repozytorium; nie wpisuj tego tekstu dosłownie. `git clone` kopiuje historię i pliki śledzone przez Git do nowego katalogu `sprawdzenie`, ale nie pobiera zawartości danych DVC. `cd sprawdzenie` przechodzi do tej kopii, a `dvc pull` odtwarza dane według wersji wskaźników zapisanej w sklonowanym commicie. W tym samym terminalu wcześniej aktywowane środowisko pozostaje aktywne; w nowym aktywuj środowisko z DVC przed `dvc pull`.

Przed klonowaniem upewnij się, że commity są dostępne pod podanym adresem (dla repozytorium zdalnego zwykle wymaga to `git push`, czyli wysłania commitów Git). To inne polecenie niż `dvc push`, które wysyła dane. Test lokalnego magazynu wymaga, aby skonfigurowana ścieżka prowadziła do dostępnego katalogu także z klonu; sam Git nie przenosi magazynu na inny komputer.

Jeżeli po sklonowaniu i wykonaniu `dvc pull` masz komplet danych — wersjonowanie działa. Jeżeli musisz komukolwiek wysłać plik ręcznie — nie działa.

## 14. Podsumowanie

Zbiór ma jednoznaczną tożsamość (suma kontrolna), opis pochodzenia (manifest), wersjonowaną zawartość (DVC), sprawdzalny kontrakt (pandera) i metryczkę odpowiadającą na pytania o prawa i ograniczenia. Podmiana jednej wartości jest wykrywana, a skasowany plik odtwarza się co do bajtu.

Czego nadal brakuje:

- kontrakt sprawdzamy ręcznie, a nie jako pierwszy krok potoku (zajęcia 4);
- przetwarzanie cech żyje w notebooku, więc trening i predykcja mogą się rozjechać (zajęcia 4);
- nie widać, które etapy trzeba przeliczyć po zmianie danych (zajęcia 4).

### Zadanie do własnego projektu

1. Dane pobierane **skryptem**, nie ręcznie — z zapisanym adresem źródła.
2. Manifest z sumą kontrolną, datą pobrania i liczbą wierszy.
3. Weryfikacja manifestu jako **pierwszy krok** potoku, zatrzymująca pracę przy niezgodności.
4. Dane pod kontrolą DVC, magazyn skonfigurowany, `dvc push` wykonany.
5. Sprawdzone odtworzenie: świeży klon + `dvc pull` daje komplet danych.
6. Kontrakt danych obejmujący typy, zakresy, braki, zamknięte zbiory kategorii i co najmniej jedną regułę międzykolumnową.
7. Raport z walidacji zapisywany jako artefakt.
8. Analiza wycieku etykiety z uzasadnieniem dla każdej podejrzanej kolumny.
9. Metryczka zbioru z pochodzeniem, licencją, kategoriami danych osobowych i znanymi ograniczeniami.

**Kryterium ukończenia:** podmiana jednej wartości w pliku danych zatrzymuje potok, a `dvc pull` po skasowaniu katalogu `data/` przywraca zbiór o tej samej sumie kontrolnej.